# DHI / EVI export from HLS
See README.md. Exports DHI GeoTIFFs for the tiles in `tiles/` to Google Drive.

In [ ]:
import ee
import dhi_evi

EE_PROJECT = "your-gee-cloud-project"   # <-- set your own Earth Engine project
TILES_GEOJSON = "tiles/proposed_additional_tif_tiles.geojson"
TILE_IDS = None                          # e.g. ["ADD03", "ADD04"]; None = all
YEARS = [2018, 2021, 2024]
DRIVE_FOLDER = "DHI_tiles"

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

tiles = dhi_evi.load_tiles(TILES_GEOJSON, TILE_IDS)
for ft in tiles:
    p = ft["properties"]
    print(p["tile_id"], p["mines"], f'{p["approx_width_km"]} x {p["approx_height_km"]} km')
print(len(tiles), "tiles x", len(YEARS), "years =", len(tiles) * len(YEARS), "tasks")

In [ ]:
submitted, errors = dhi_evi.run_exports(tiles, YEARS, DRIVE_FOLDER)

## Check for failed tasks (run after the tasks finish)

In [ ]:
ids = {ft["properties"]["tile_id"] for ft in tiles}
failed = dhi_evi.failed_tasks(ids, YEARS)
for (tid, yr), msg in sorted(failed.items()):
    print(tid, yr, msg[:80])
print(len(failed), "failed")

In [ ]:
# Re-submit failed tile/years
by_id = {ft["properties"]["tile_id"]: ft for ft in tiles}
for (tid, yr) in sorted(failed):
    print(dhi_evi.export_tile_year(tid, yr, dhi_evi.build_roi(by_id[tid]), DRIVE_FOLDER))